# Streamed DPO on a free Colab T4: does this run ship?

**This is the sequence of cells as it was run on a free Colab T4 on 2026-09-30**, including the run that failed and
the workaround. The browser tab that held the cell outputs was closed before the notebook was saved, so this file
has no outputs. They are not lost: every phase goes through `scripts/run_logged.sh`, which wrote the full output to
`logs/t4/<phase>.log` (UTC-timestamped) with its exit code in `logs/t4/<phase>.exit`, and GPU samples to
`nvidia-smi/<phase>.csv` and `nvidia-smi/<phase>_full.txt`. Each section names the phase logs that hold its output.

Runtime -> Change runtime type -> **T4 GPU**.

## 0. Machine  (`nvidia-smi/00_start.txt`, `logs/t4/00_host.txt`)

In [ ]:
!git clone -q https://github.com/abdulwaarith0/soup-dpo-t4-take-home.git
%cd soup-dpo-t4-take-home
!mkdir -p logs/t4 nvidia-smi
!nvidia-smi | tee nvidia-smi/00_start.txt
!(date -u; free -h; df -h /; nproc; cat /proc/cpuinfo | grep 'model name' | head -1) | tee logs/t4/00_host.txt

## 1. Install  (`01a_*`, `01_versions.txt`)

Colab's Python is 3.13 and every soup-cli release from 0.73.0 up declares `Requires-Python <3.13`, so the task's own
`pip install "soup-cli[train]"` resolves to 0.72.4 here. The first line records that without installing it; the rest
installs 0.75.1, the latest release, in a Python 3.12 venv that every later cell uses.

In [ ]:
!python --version | tee logs/t4/01a_colab_python.txt
!bash scripts/run_logged.sh 01a_unpinned_install_dry_run -- pip install --dry-run "soup-cli[train]"
!pip install -q uv && uv venv -q --python 3.12 /content/venv && uv pip install -q --python /content/venv/bin/python "soup-cli[train]==0.75.1"
import os
os.environ["PATH"] = "/content/venv/bin:" + os.environ["PATH"]
os.environ["VIRTUAL_ENV"] = "/content/venv"
!which python soup
!python -c "import torch,transformers,peft,trl,accelerate,soup_cli,sys;print('python',sys.version.split()[0]);print('torch',torch.__version__,'cuda',torch.version.cuda);print('transformers',transformers.__version__);print('peft',peft.__version__);print('trl',trl.__version__);print('accelerate',accelerate.__version__);print('soup-cli',soup_cli.__version__);print('bf16 native:',torch.cuda.is_bf16_supported(including_emulation=False),'| bf16 incl. emulation:',torch.cuda.is_bf16_supported())" | tee logs/t4/01_versions.txt

## 2. Data: rebuild, audit, Soup's own data checks  (`02_*` to `06_*`)

In [ ]:
!bash scripts/run_logged.sh 02_prepare_data -- python data/prepare_data.py
!bash scripts/run_logged.sh 03_audit_data -- python scripts/audit_data.py --data data/train.jsonl --max-length 1024
!bash scripts/run_logged.sh 04_soup_data_lint -- soup data lint data/train.jsonl
!bash scripts/run_logged.sh 05_soup_data_validate -- soup data validate data/train.jsonl
!bash scripts/run_logged.sh 06_soup_data_doctor -- soup data doctor data/train.jsonl --model Qwen/Qwen2.5-1.5B-Instruct

## 3. Memory budget BEFORE training, soup doctor, dry run  (`07_*` to `09_*`)

In [ ]:
!bash scripts/run_logged.sh 07_memory_budget -- python scripts/memory_budget.py --batch 2 --seq 1024
!bash scripts/run_logged.sh 08_soup_doctor -- soup doctor
!bash scripts/run_logged.sh 09_dry_run -- soup train --config configs/dpo_stream_t4.yaml --dry-run

## 4. The run under review

First attempt (`10_train_main.log`, exit 1): stopped at setup with `cached snapshot file 'model.safetensors' points
outside the Hugging Face blob store`. The diagnosis cell shows why: the snapshot's blob is itself a link into a shared
`hub/blobs/<xx>/` store, which Soup's snapshot check (`utils/spectrum_scan.py:690-696`) refuses.

In [ ]:
!nohup bash scripts/run_logged.sh 10_train_main -- python scripts/train_with_mem.py --config configs/dpo_stream_t4.yaml --yes --probe > /dev/null 2>&1 &

In [ ]:
# Diagnose the setup failure: full traceback, HF env, and where the cached snapshot's weights link to.
!grep -n -A60 "Traceback (most recent call last)" logs/t4/10_train_main.log | cut -c30-230 | head -90
!env | grep -i -E "^HF_|HUGGINGFACE|TRANSFORMERS_CACHE|XDG_CACHE"
!ls -la ~/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/*/ 2>&1 | head -20
!readlink -f ~/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/*/model.safetensors

In [ ]:
# Workaround: the same weights, same revision, as regular files in a plain directory; only `base:` changes,
# in Colab-only config copies (logs/t4/10a_*).
!ls ~/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/ | tee logs/t4/10a_model_revision.txt
!bash scripts/run_logged.sh 10a_local_copy_of_base -- python -c "import os; from huggingface_hub import snapshot_download; rev=open('logs/t4/10a_model_revision.txt').read().split()[0]; p=snapshot_download('Qwen/Qwen2.5-1.5B-Instruct', revision=rev, local_dir='/content/models/Qwen2.5-1.5B-Instruct'); print(p, rev); print('symlinks:', sum(os.path.islink(os.path.join(p, f)) for f in os.listdir(p)))"
!for c in dpo_stream_t4 control_recipe_lr; do sed "s#^base: .*#base: /content/models/Qwen2.5-1.5B-Instruct#" configs/$c.yaml > configs/colab_$c.yaml; done; diff configs/dpo_stream_t4.yaml configs/colab_dpo_stream_t4.yaml

In [ ]:
# The run under review, retried with the only change being the plain-directory base (logs/t4/10b_train_main.log).
!nohup bash scripts/run_logged.sh 10b_train_main -- python scripts/train_with_mem.py --config configs/colab_dpo_stream_t4.yaml --yes --probe > /dev/null 2>&1 &

In [ ]:
# Progress: pre-flight panel and logits probe, then per step: loss, grad_norm (nan = fp16 overflow, step skipped), lr, margin, accuracy
!grep -E "base store|VRAM|forecast|streaming ready|LoRA applied|LOGITS_PROBE|MEASURED|Error|Traceback|train_runtime" logs/t4/10b_train_main.log | cut -c1-260 | tail -20
!grep -o "{'loss'.*}" logs/t4/10b_train_main.log | python -c "import sys,ast; [print(i+1, d['loss'], d['grad_norm'], d['learning_rate'], d.get('rewards/margins'), d.get('rewards/accuracies')) for i,d in enumerate(map(ast.literal_eval, sys.stdin))]" | tail -70
!cat logs/t4/10b_train_main.exit 2>/dev/null || echo "(still running)"

## 5. Verification with null and random controls, first pass rule  (`11_verify.log`, `runs/verify_main.json`)

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"  # verify + ship read the cached snapshot 989aa79, the same revision the run trained on
!bash scripts/run_logged.sh 11_verify -- python scripts/verify_training.py --adapter runs/main --dtype float32

## 6. `soup ship` on the trained adapter  (`12_ship_main.log`, `runs/ship_main_*.json`, about 41 minutes)

In [ ]:
!bash scripts/run_logged.sh 12_ship_main -- soup ship --base Qwen/Qwen2.5-1.5B-Instruct --adapter runs/main --task-eval eval/task_eval_ru.jsonl --config configs/dpo_stream_t4.yaml --device cuda --emit-evidence runs/ship_main_evidence.json --output runs/ship_main_verdict.json

## 7. Control run: the catalog recipes' learning rate, 5e-6  (`15_*`, `16_*`)

Same config and data; only `lr`. It passed the first pass rule by 0.002 while scoring 0.26 on chosen-shorter pairs,
which exposed that rule's flaw (see the `scripts/verify_training.py` docstring).

In [ ]:
!nohup bash scripts/run_logged.sh 15_train_control_recipe_lr -- python scripts/train_with_mem.py --config configs/colab_control_recipe_lr.yaml --yes > /dev/null 2>&1 &

In [ ]:
!grep -E "peak VRAM|free VRAM|streaming ready|LoRA applied|MEASURED|Error|Traceback|train_runtime" logs/t4/15_train_control_recipe_lr.log | cut -c1-260 | tail -12
!grep -o "{'loss'.*}" logs/t4/15_train_control_recipe_lr.log | python -c "import sys,ast; [print(i+1, d['loss'], d['grad_norm'], d['learning_rate'], d.get('rewards/margins'), d.get('rewards/accuracies')) for i,d in enumerate(map(ast.literal_eval, sys.stdin))]" | tail -70
!cat logs/t4/15_train_control_recipe_lr.exit 2>/dev/null || echo "(still running)"

In [ ]:
!bash scripts/run_logged.sh 16_verify_control_recipe_lr -- python scripts/verify_training.py --adapter runs/control_recipe_lr --dtype float32 --no-controls

## 7b. The fix the verdict asks for, tested  (`17_train_fixed.log`)

In [ ]:
# The main config with only the data changed (data/prepare_fixed_data.py: conversational Qwen format + 250/250
# length-balanced, same 100 held-out pairs excluded).
!git pull -q && sed "s#^base: .*#base: /content/models/Qwen2.5-1.5B-Instruct#" configs/fixed_data.yaml > configs/colab_fixed_data.yaml && diff configs/dpo_stream_t4.yaml configs/fixed_data.yaml; sha256sum data/train_fixed.jsonl | cut -c1-16
!nohup bash scripts/run_logged.sh 17_train_fixed -- python scripts/train_with_mem.py --config configs/colab_fixed_data.yaml --yes > /dev/null 2>&1 &

In [ ]:
!grep -E "peak VRAM|streaming ready|LoRA applied|MEASURED|Error|Traceback|train_runtime" logs/t4/17_train_fixed.log | cut -c1-240 | tail -8
!grep -o "{'loss'.*}" logs/t4/17_train_fixed.log | python -c "import sys,ast; [print(i+1, d['loss'], d['grad_norm'], d['learning_rate'], d.get('rewards/margins'), d.get('rewards/accuracies')) for i,d in enumerate(map(ast.literal_eval, sys.stdin))]" | tail -70
!cat logs/t4/17_train_fixed.exit 2>/dev/null || echo "(still running)"

## 7c. Verification with the length-balanced rule, all runs  (`18_*` to `20_*`, `runs/verify_*_v2.json`, `runs/verify_fixed.json`)

In [ ]:
!git pull -q && git log --oneline -1
!bash scripts/run_logged.sh 18_verify_fixed -- python scripts/verify_training.py --adapter runs/fixed --train data/train_fixed.jsonl --dtype float32 --no-controls --out runs/verify_fixed.json
!bash scripts/run_logged.sh 19_verify_main_v2 -- python scripts/verify_training.py --adapter runs/main --dtype float32 --out runs/verify_main_v2.json
!bash scripts/run_logged.sh 20_verify_control_recipe_lr_v2 -- python scripts/verify_training.py --adapter runs/control_recipe_lr --dtype float32 --no-controls --out runs/verify_control_recipe_lr_v2.json

## 8. Package everything (this zip is what was committed under `logs/`, `nvidia-smi/` and `runs/`)

In [ ]:
!cd /content/soup-dpo-t4-take-home && cat logs/t4/20_verify_control_recipe_lr_v2.exit && nvidia-smi > nvidia-smi/99_end.txt && rm -f t4_outputs.zip && zip -q -r t4_outputs.zip logs nvidia-smi runs/*.json configs/colab_*.yaml runs/main/adapter_config.json runs/main/adapter_model.safetensors runs/main/ref runs/control_recipe_lr/adapter_config.json runs/control_recipe_lr/adapter_model.safetensors runs/fixed/adapter_config.json runs/fixed/adapter_model.safetensors -x "*/checkpoint-*" && ls -la t4_outputs.zip

In [ ]:
from google.colab import files; files.download('/content/soup-dpo-t4-take-home/t4_outputs.zip')